In [2]:
# Import library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Setup pandas
pd.set_option('display.max_columns', None)

# Get the current working directory
os.getcwd()

'/Users/ahmadyusufalbadri/Documents/1 - Learning/4 - Competition/datavidia 2025/Datavidia'

# EDA

The goal of this EDA are:

1. To find relationship between agri-food factors to the global warming from the training data.
2. Find possible feature imputation and engineering for the machine learning model.

## Initial Data Formatting

In [8]:
# Get the data
df_lat_lon_country = pd.read_csv('data/lat_lon_data_pul2.csv')
df_lat_lon_country = df_lat_lon_country.iloc[:, 1:]

df_train = pd.read_csv('data/train.csv')
df_train[['Negara', 'Tahun']] = df_train['Negara/Tahun'].str.split('/', expand=True)
df_train = df_train.iloc[:, 2:]
df_train = df_train.merge(df_lat_lon_country, how='left', left_on='Negara', right_on='country')
df_train.drop('country', axis=1, inplace=True)
df_train.head()

,Emisi Savanna Api,Emisi Kebakaran Hutan,Emisi Residu Tanaman,Emisi Budidaya Padi,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Konsumsi Rumah Tangga Makanan,Emisi Ritel Makanan,Emisi Penggunaan Listrik Di Pertanian,Emisi Kemasan Makanan,Emisi Sistem Agrifood Pembuangan Limbah,Emisi Pengolahan Makanan,Emisi Manufaktur Pupuk,Emisi IPPU,Emisi Kotoran Diterapkan Pada Tanah,Emisi Pupuk Kandang Di Padang Rumput,Emisi Manajemen Pupuk,Emisi Kebakaran Di Tanah Organik,Emisi Kebakaran Di Hutan Tropis Yang Lembab,Penggunaan Energi Di Pertanian,Populasi Pedesaan,Populasi Perkotaan,Total Populasi - Pria,Total Populasi - Wanita,Emisi Total,Peningkatan Suhu Rata - Rata ° C,Negara,Tahun,lat,lon
0,14.7237,0.0557,205.6077,686.00,0.0,11.807483,63.1152,-2388.803,0.0,79.0851,109.6446,14.2666,67.631366,691.7888,252.21419,11.9970,209.9778,260.1431,1590.5319,319.1763,0.0,0.0,NaN,9655167.0,2593947.0,5348387.0,5346409.0,2198.963539,0.536167,Afghanistan,1990,33.768006,66.238514
1,14.7237,0.0557,209.4971,678.16,0.0,11.712073,61.2125,-2388.803,0.0,80.4885,116.6789,11.4182,67.631366,710.8212,252.21419,12.8539,217.0388,268.6292,1657.2364,342.3079,0.0,0.0,NaN,10230490.0,2763167.0,5372959.0,5372208.0,2323.876629,0.020667,Afghanistan,1991,33.768006,66.238514
2,14.7237,0.0557,196.5341,686.00,0.0,11.712073,53.3170,-2388.803,0.0,80.7692,126.1721,9.2752,67.631366,743.6751,252.21419,13.4929,222.1156,264.7898,1653.5068,349.1224,0.0,0.0,NaN,10995568.0,2985663.0,6028494.0,6028939.0,2356.304229,-0.259583,Afghanistan,1992,33.768006,66.238514
3,14.7237,0.0557,230.8175,686.00,0.0,11.712073,54.3617,-2388.803,0.0,85.0678,81.4607,9.0635,67.631366,791.9246,252.21419,14.0559,201.2057,261.7221,1642.9623,352.2947,0.0,0.0,NaN,11858090.0,3237009.0,7003641.0,7000119.0,2368.470529,0.101917,Afghanistan,1993,33.768006,66.238514
4,14.7237,0.0557,242.0494,705.60,0.0,11.712073,53.9874,-2388.803,0.0,88.8058,90.4008,8.3962,67.631366,831.9181,252.21419,15.1269,182.2905,267.6219,1689.3593,367.6784,0.0,0.0,NaN,12690115.0,3482604.0,7733458.0,7722096.0,2500.768729,0.372250,Afghanistan,1994,33.768006,66.238514


In [16]:
# Repositioned the data
temp = df_train.columns.to_list()[-4:] + [df_train.columns.to_list()[-5]] + df_train.columns.to_list()[:-5]
df_train = df_train[temp]
df_train.head()

,Negara,Tahun,lat,lon,Peningkatan Suhu Rata - Rata ° C,Emisi Savanna Api,Emisi Kebakaran Hutan,Emisi Residu Tanaman,Emisi Budidaya Padi,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Konsumsi Rumah Tangga Makanan,Emisi Ritel Makanan,Emisi Penggunaan Listrik Di Pertanian,Emisi Kemasan Makanan,Emisi Sistem Agrifood Pembuangan Limbah,Emisi Pengolahan Makanan,Emisi Manufaktur Pupuk,Emisi IPPU,Emisi Kotoran Diterapkan Pada Tanah,Emisi Pupuk Kandang Di Padang Rumput,Emisi Manajemen Pupuk,Emisi Kebakaran Di Tanah Organik,Emisi Kebakaran Di Hutan Tropis Yang Lembab,Penggunaan Energi Di Pertanian,Populasi Pedesaan,Populasi Perkotaan,Total Populasi - Pria,Total Populasi - Wanita,Emisi Total
0,Afghanistan,1990,33.768006,66.238514,0.536167,14.7237,0.0557,205.6077,686.00,0.0,11.807483,63.1152,-2388.803,0.0,79.0851,109.6446,14.2666,67.631366,691.7888,252.21419,11.9970,209.9778,260.1431,1590.5319,319.1763,0.0,0.0,NaN,9655167.0,2593947.0,5348387.0,5346409.0,2198.963539
1,Afghanistan,1991,33.768006,66.238514,0.020667,14.7237,0.0557,209.4971,678.16,0.0,11.712073,61.2125,-2388.803,0.0,80.4885,116.6789,11.4182,67.631366,710.8212,252.21419,12.8539,217.0388,268.6292,1657.2364,342.3079,0.0,0.0,NaN,10230490.0,2763167.0,5372959.0,5372208.0,2323.876629
2,Afghanistan,1992,33.768006,66.238514,-0.259583,14.7237,0.0557,196.5341,686.00,0.0,11.712073,53.3170,-2388.803,0.0,80.7692,126.1721,9.2752,67.631366,743.6751,252.21419,13.4929,222.1156,264.7898,1653.5068,349.1224,0.0,0.0,NaN,10995568.0,2985663.0,6028494.0,6028939.0,2356.304229
3,Afghanistan,1993,33.768006,66.238514,0.101917,14.7237,0.0557,230.8175,686.00,0.0,11.712073,54.3617,-2388.803,0.0,85.0678,81.4607,9.0635,67.631366,791.9246,252.21419,14.0559,201.2057,261.7221,1642.9623,352.2947,0.0,0.0,NaN,11858090.0,3237009.0,7003641.0,7000119.0,2368.470529
4,Afghanistan,1994,33.768006,66.238514,0.372250,14.7237,0.0557,242.0494,705.60,0.0,11.712073,53.9874,-2388.803,0.0,88.8058,90.4008,8.3962,67.631366,831.9181,252.21419,15.1269,182.2905,267.6219,1689.3593,367.6784,0.0,0.0,NaN,12690115.0,3482604.0,7733458.0,7722096.0,2500.768729


## Analysis

In [17]:
# Check data information
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5603 entries, 0 to 5602
Data columns (total 33 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Negara                                       5603 non-null   object 
 1   Tahun                                        5603 non-null   object 
 2   lat                                          5603 non-null   float64
 3   lon                                          5603 non-null   float64
 4   Peningkatan Suhu Rata - Rata ° C             5603 non-null   float64
 5   Emisi Savanna Api                            5578 non-null   float64
 6   Emisi Kebakaran Hutan                        5528 non-null   float64
 7   Emisi Residu Tanaman                         4485 non-null   float64
 8   Emisi Budidaya Padi                          5603 non-null   float64
 9   Emisi Tanah Organik Yang Dikeringkan (Co2)   5603 non-null   float64
 10  

In [21]:
# Check missing values and duplicates
print('====================================\nMISSING VALUES CHECK:\n====================================')
print(df_train.isna().sum().sort_values(ascending=False))
print()
print('====================================\nDATA DUPLICATE CHECK:', df_train.duplicated().sum(), '\n====================================')

MISSING VALUES CHECK:
Emisi Residu Tanaman                           1118
Penggunaan Energi Di Pertanian                  782
Emisi Kotoran Diterapkan Pada Tanah             752
Emisi Manajemen Pupuk                           752
Emisi IPPU                                      599
Lahan Hutan                                     397
Konversi Hutan Bersih                           397
Emisi Konsumsi Rumah Tangga Makanan             389
Emisi Kebakaran Di Hutan Tropis Yang Lembab     125
Emisi Kebakaran Hutan                            75
Emisi Savanna Api                                25
Populasi Perkotaan                                0
Populasi Pedesaan                                 0
Emisi Manufaktur Pupuk                            0
Emisi Kebakaran Di Tanah Organik                  0
Total Populasi - Pria                             0
Emisi Pupuk Kandang Di Padang Rumput              0
Total Populasi - Wanita                           0
Negara                                    

In [22]:
# Convert tahun to int
df_train['Tahun'] = df_train['Tahun'].astype(int)

In [24]:
# Describe the data
df_train.describe().astype(str)

,Tahun,lat,lon,Peningkatan Suhu Rata - Rata ° C,Emisi Savanna Api,Emisi Kebakaran Hutan,Emisi Residu Tanaman,Emisi Budidaya Padi,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Konsumsi Rumah Tangga Makanan,Emisi Ritel Makanan,Emisi Penggunaan Listrik Di Pertanian,Emisi Kemasan Makanan,Emisi Sistem Agrifood Pembuangan Limbah,Emisi Pengolahan Makanan,Emisi Manufaktur Pupuk,Emisi IPPU,Emisi Kotoran Diterapkan Pada Tanah,Emisi Pupuk Kandang Di Padang Rumput,Emisi Manajemen Pupuk,Emisi Kebakaran Di Tanah Organik,Emisi Kebakaran Di Hutan Tropis Yang Lembab,Penggunaan Energi Di Pertanian,Populasi Pedesaan,Populasi Perkotaan,Total Populasi - Pria,Total Populasi - Wanita,Emisi Total
count,5603.0,5603.0,5603.0,5603.0,5578.0,5528.0,4485.0,5603.0,5603.0,5603.0,5603.0,5206.0,5206.0,5214.0,5603.0,5603.0,5603.0,5603.0,5603.0,5603.0,5004.0,4851.0,5603.0,4851.0,5603.0,5478.0,4821.0,5603.0,5603.0,5603.0,5603.0,5603.0
mean,2002.1167231840086,17.642178043899094,9.308831156629843,0.7501702568951026,1229.6544452491933,924.2751989146165,949.2050756075809,4241.101703878332,3471.7915703908625,327.5222891339001,1823.1383028019652,-18145.65406171725,18454.522389684982,4290.229335558112,1726.5583187622726,1524.6925477490424,1547.357603083276,5919.614031722235,3779.3597751397306,2900.800427282518,17309.192334972024,913.2562305916307,3431.750042752865,2252.8005198309625,1254.336085041337,668.2839432822198,3015.8050467537855,17949028.825629126,15738814.348384794,17038810.178604193,16756698.563587843,60571.36914473102
std,7.166376713604766,24.178537713251004,76.06667760094686,0.4989433758492056,5472.353827553386,3692.5848811941282,3498.477163822186,17599.627762521446,15488.661938100191,1445.4848650654835,5306.417931085816,81045.73582102315,108004.96140860341,20691.80766340678,6882.576637017679,8356.445934711754,10235.123944206767,21758.44409807351,19356.823253121755,11016.456584276986,91075.90974998647,3215.0607254423044,8926.859188341768,8010.868244206085,23189.667834312146,3243.015188127321,12391.323705683406,90736574.0355425,59203131.67387243,73982191.22445323,70561958.84090337,208490.41680261333
min,1990.0,-51.9492937,-176.204224,-1.4158333333333335,0.0,0.0,0.0002,0.0,0.0,0.0,0.0001,-797183.079,0.0,0.0,0.0,0.0,0.0,0.34,0.0001,0.0019,0.0,0.049,0.0007,0.4329,0.0,0.0,0.0319,0.0,0.0,270.0,300.0,-391884.0563
25%,1996.0,1.613172,-60.9620777,0.44033333333333335,0.0,0.0,10.9211,172.2193,0.0,5.0,26.262663446183105,-3224.3715,0.0,10.011225,20.42625,7.2017500000000005,67.63136607385836,85.68311232412132,209.58772835754152,356.4910168671663,34.17885,15.263200000000001,142.52530000000002,36.8658,0.0,0.0,12.1815,94314.0,204572.0,194837.0,200354.5,5135.02952071779
50%,2002.0,16.7417041,14.12456,0.72975,1.6579,0.66185,98.0374,515.5172,0.0,12.328280513854102,176.0653,-69.5346,44.616,136.2674,141.7096,27.636841710447985,71.28785384639453,868.0077,332.6036474920459,1093.665,707.7274,113.5816,964.8423,255.4619,0.0,0.0,134.4775,1572038.0,2201558.0,2336340.0,2338950.0,11748.019463854458
75%,2008.0,38.6281733,47.9734174,1.0301666666666667,114.3092,71.0949,356.2161,1551.41055,694.1645000000001,106.9817314214948,1135.64975,0.0,4734.96775,1357.8258999999998,905.57545,492.7371464768823,267.07297555428045,2945.4187,1108.6982,2012.447178005524,5675.145625,454.5236,2319.68205,1121.8815,0.0,10.189775,1151.3962,7836860.5,7690343.0,8586905.5,8617357.5,32921.11201448267
max,2014.0,64.9841821,179.1582918181797,3.143,114616.4011,52227.6306,30638.5338,164915.2556,232118.4694,16459.0,62048.1673,171121.076,1605106.096,344626.3392,104356.5616,141904.3336,175741.3061,213289.7016,274253.5125,170826.4233,1619276.0171,34677.3603,88494.2199,70592.6465,991717.5431,51771.2568,248879.1769,874857621.0,779954516.0,724363944.0,692204586.0,2741629.5747


In [39]:
# Check countries with null
temp = df_train.isna().sum().sort_values(ascending=False)
temp2 = df_train['Negara'].value_counts(dropna=False).reset_index()
for i in temp[temp > 0].index:
    print(f'NULL VALUES FOR __{i}__ AS FOLLOW:')
    print(df_train[df_train[i].isna()]['Negara'].value_counts(dropna=False).reset_index().merge(temp2, how='left', on='Negara'))
    print()

NULL VALUES FOR __Emisi Residu Tanaman__ AS FOLLOW:
                                          Negara  count_x  count_y
0                                 American Samoa       25       25
1                                  Liechtenstein       25       25
2                                        Mayotte       25       25
3                                         Monaco       25       25
4                                          Nauru       25       25
5                                           Niue       25       25
6   Saint Helena, Ascension and Tristan da Cunha       25       25
7                                    Saint Lucia       25       25
8                      Saint Pierre and Miquelon       25       25
9                                          Samoa       25       25
10                                    San Marino       25       25
11                                    Seychelles       25       25
12                                     Singapore       25       25
13        

In [40]:
# Check a sample of Bahrain
df_train[df_train['Negara'] == 'Bahrain']

,Negara,Tahun,lat,lon,Peningkatan Suhu Rata - Rata ° C,Emisi Savanna Api,Emisi Kebakaran Hutan,Emisi Residu Tanaman,Emisi Budidaya Padi,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Konsumsi Rumah Tangga Makanan,Emisi Ritel Makanan,Emisi Penggunaan Listrik Di Pertanian,Emisi Kemasan Makanan,Emisi Sistem Agrifood Pembuangan Limbah,Emisi Pengolahan Makanan,Emisi Manufaktur Pupuk,Emisi IPPU,Emisi Kotoran Diterapkan Pada Tanah,Emisi Pupuk Kandang Di Padang Rumput,Emisi Manajemen Pupuk,Emisi Kebakaran Di Tanah Organik,Emisi Kebakaran Di Hutan Tropis Yang Lembab,Penggunaan Energi Di Pertanian,Populasi Pedesaan,Populasi Perkotaan,Total Populasi - Pria,Total Populasi - Wanita,Emisi Total
371,Bahrain,1990,26.030093,50.553337,0.652750,0.0,0.0,0.0006,246.407276,0.0,0.0,116.0378,-1.8626,0.0,181.3300,154.9085,21.1401,625.5804,320.7606,247.314516,70.4939,4237.9918,0.6294,12.2198,1.5460,0.0,0.0,NaN,58816.0,437115.0,302861.0,214558.0,6234.498093
372,Bahrain,1991,26.030093,50.553337,-0.267417,0.0,0.0,0.0006,246.407276,0.0,1.0,117.1316,-1.8626,0.0,176.0500,138.9684,12.8610,601.4023,331.0686,209.587728,86.6473,1731.5459,0.6363,11.9063,1.5451,0.0,0.0,NaN,59494.0,450271.0,313375.0,222040.0,3664.895805
373,Bahrain,1992,26.030093,50.553337,-0.679583,0.0,0.0,0.0010,246.407276,0.0,0.0,129.9014,-1.8626,0.0,206.0455,144.2447,17.3818,620.8618,340.3253,209.587728,101.1022,1944.8331,0.6505,12.0762,1.5533,0.0,0.0,NaN,60688.0,462399.0,324822.0,229648.0,3973.109205
374,Bahrain,1993,26.030093,50.553337,0.239417,0.0,0.0,0.0013,246.407276,0.0,0.0,125.2621,-1.8626,0.0,205.0604,118.9537,18.1512,758.5181,348.4510,209.587728,114.5978,2337.2729,0.6608,11.8851,1.5549,0.0,0.0,NaN,62228.0,473985.0,336425.0,237331.0,4494.501705
375,Bahrain,1994,26.030093,50.553337,0.822167,0.0,0.0,0.0015,246.407276,0.0,0.0,132.0274,-1.8626,0.0,230.1559,149.0632,21.7109,757.7321,355.8552,209.587728,132.1606,2112.1978,0.6677,11.5924,1.5412,0.0,0.0,NaN,63798.0,485790.0,348166.0,245088.0,4358.838305
376,Bahrain,1995,26.030093,50.553337,0.282750,0.0,0.0,0.0006,246.407276,0.0,0.0,134.1660,-1.8626,0.0,242.7003,175.2565,22.2992,797.3384,362.3908,209.587728,147.2936,2147.4887,0.6624,10.9094,1.4645,0.0,0.0,NaN,65454.0,498245.0,360022.0,252912.0,4496.102805
377,Bahrain,1996,26.030093,50.553337,0.854667,0.0,0.0,0.0003,246.407276,0.0,0.0,142.3140,-1.8626,0.0,255.8283,160.2765,23.0986,828.7693,372.6832,209.587728,164.8404,2055.0058,0.6836,11.6595,1.5715,0.0,0.0,NaN,67211.0,511457.0,371964.0,260792.0,4470.863405
378,Bahrain,1997,26.030093,50.553337,0.310583,0.0,0.0,0.0002,246.407276,0.0,0.0,149.4938,-1.8626,0.0,256.3521,187.5107,24.3320,549.2585,382.2813,209.587728,183.6778,2255.8428,0.7010,11.7978,1.5941,0.0,0.0,NaN,69119.0,525811.0,383949.0,268694.0,4456.974505
379,Bahrain,1998,26.030093,50.553337,1.682667,0.0,0.0,0.0002,246.407276,0.0,0.0,157.7614,-1.8626,0.0,286.0472,223.2288,30.4988,191.0169,391.2334,209.587728,201.8677,2194.0055,0.5460,11.3571,1.4136,0.0,0.0,NaN,71319.0,542383.0,395916.0,276567.0,4143.109005
380,Bahrain,1999,26.030093,50.553337,1.823333,0.0,0.0,0.0002,246.407276,0.0,0.0,163.9440,-1.8626,0.0,302.5149,269.1444,28.5776,236.7943,400.0025,209.587728,222.0003,2242.1034,0.5443,11.8070,1.4711,0.0,0.0,NaN,73994.0,562551.0,407807.0,284326.0,4333.036405
